# Baseline B0: DINOv2 ViT-B/14 + GeM (+ Lw) on ROxford5k / RParis6k

Same pipeline as the original GeM (Radenović et al., TPAMI 2018); only the backbone changes from ResNet101 → **DINOv2 ViT-B/14**:

`image → DINOv2 patch tokens → ReLU → GeM (learnable p) → L2 → [Lw whitening] → L2 → kNN`

- **Stage 1, B0-frozen:** no training. Quick sanity check against the DINOv2 paper (Table 9, ViT-B/14 off-the-shelf: ROxf M/H ≈ 72.9/49.5, RPar M/H ≈ 90.3/78.6; the protocol differs slightly, so it is only a rough reference).
- **Stage 2, B0-finetune:** fine-tune the last few blocks on retrieval-SfM-120k with contrastive loss + hard-negative mining (2000 queries, pool 22k, 5 negatives), as in the original GeM.
- **Evaluation:** ROxf/RPar Easy/Medium/Hard, mAP + mP@{1,5,10}; single-scale and multi-scale `[1, 1/√2, 1/2]`; with and without Lw. No αQE, no re-ranking.

## How to run on Kaggle
1. **Settings → Accelerator: GPU T4 (or P100)**. The notebook uses a single GPU.
2. **Settings → Internet: On** (required to download the datasets and DINOv2 weights). The Kaggle account must be phone-verified to enable Internet.
3. Test run: **Run All** with `RUN_STAGE2 = False` to get Stage 1 numbers first (about 1 hour, estimated).
4. Full run: **Save Version → Save & Run All (Commit)**. A commit runs in the background for up to 12 hours, so the browser can be closed. Results are in the *Output* tab: `b0/results.csv`, `b0/results.json`, checkpoint `b0/ckpt_last.pt`.

Data is downloaded to `/tmp/cbir_data` (scratch disk, not limited by the 20 GB of `/kaggle/working`). The notebook prints free disk space before downloading.

> All timings in this notebook are **estimates**, not yet measured on a real Kaggle GPU.

In [ ]:
# ===================== CONFIG: edit here =====================
import math
RUN_STAGE1 = False         # B0-frozen: no training (set True for comparison numbers)
RUN_STAGE2 = True          # B0-finetune: train on rSfM-120k
LEARN_WHITEN = True        # learn Lw (needs rSfM-120k download, tens of GB)

CFG = dict(
    data_root   = '/tmp/cbir_data',        # scratch disk, larger than /kaggle/working
    out_dir     = '/kaggle/working/b0',    # results + checkpoints (shown in the Output tab)
    arch        = 'dinov2_vitb14',
    datasets    = ['roxford5k', 'rparis6k'],
    test_imsize = 518,                     # longest side, rounded down to a multiple of 14
    multiscale  = [1, 1/2**0.5, 1/2],
    gem_p       = 3.0,
    whiten_max_pairs = 5000,               # number of q/p pairs used to learn Lw (all pairs is very slow)
    # ---- training (stage 2) ----
    train_imsize    = 518,
    finetune_blocks = 2,                   # number of last ViT blocks to train (ViT-B has 12)
    epochs          = 10,
    lr              = 2e-6,                # Adam; original GeM uses 5e-7 for R101, ViT needs tuning
    p_lr_mult       = 10,                  # lr of p = lr * 10 (as in the original code)
    weight_decay    = 1e-6,
    lr_gamma        = math.exp(-0.01),     # ExponentialLR as in the original code
    margin          = 0.85,
    query_size      = 2000,
    pool_size       = 22000,
    neg_num         = 5,
    batch_size      = 5,                   # tuples per optimizer step
    max_train_hours = 8.0,                 # stop training before the 12-hour session limit
    eval_every_epoch = False,              # True: evaluate ROxf single-scale after each epoch (~5 extra min/epoch)
    resume_path     = '',                  # path to ckpt_last.pt to resume training (e.g. from a previous Output)
    num_workers = 4,
    amp         = True,                    # fp16 autocast
    seed        = 0,
)

In [ ]:
import os, sys, gc, time, json, pickle, shutil, subprocess, random, glob
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm

random.seed(CFG['seed']); np.random.seed(CFG['seed']); torch.manual_seed(CFG['seed'])
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = CFG['amp'] and DEV.type == 'cuda'
os.makedirs(CFG['data_root'], exist_ok=True); os.makedirs(CFG['out_dir'], exist_ok=True)

print('torch', torch.__version__, '| device', DEV, torch.cuda.get_device_name(0) if DEV.type == 'cuda' else '')
def disk(path):
    u = shutil.disk_usage(path); return f'{path}: free {u.free/2**30:.1f} GB / total {u.total/2**30:.1f} GB'
for p in [CFG['data_root'], CFG['out_dir']]: print(disk(p))
if DEV.type != 'cuda': print('!! No GPU found: go to Settings → Accelerator → GPU')

## 1. Download data (Oxford/Paris + revisited ground truth; rSfM-120k for Lw and training)

In [ ]:
def sh(cmd):
    print('$', cmd); subprocess.run(cmd, shell=True, check=True)

def remote_size_gb(url):
    try:
        out = subprocess.run(f'curl -sIL --max-time 30 "{url}"', shell=True, capture_output=True, text=True).stdout
        sizes = [int(l.split(':')[1]) for l in out.splitlines() if l.lower().startswith('content-length')]
        return sizes[-1] / 2**30 if sizes else None
    except Exception:
        return None

def fetch(url, dst):
    if not os.path.exists(dst):
        try: sh(f'wget -q --tries=5 -O "{dst}.part" "{url}" && mv "{dst}.part" "{dst}"')
        except subprocess.CalledProcessError:
            if os.path.exists(dst + '.part'): os.remove(dst + '.part')
            raise RuntimeError(f'Download failed: {url} (check Internet: On, or the link has changed)')

def download_test(root):
    # same as cirtorch/utils/download.py (download_test)
    src = {'oxford5k': ('https://www.robots.ox.ac.uk/~vgg/data/oxbuildings', ['oxbuild_images-v1.tgz']),
           'paris6k':  ('https://www.robots.ox.ac.uk/~vgg/data/parisbuildings', ['paris_1-v1.tgz', 'paris_2-v1.tgz'])}
    for ds, (base, files) in src.items():
        jpg = f'{root}/test/{ds}/jpg'
        if os.path.isdir(jpg) and len(os.listdir(jpg)) > 1000: continue
        tmp = f'{root}/test/{ds}/tmp'; os.makedirs(tmp, exist_ok=True); os.makedirs(jpg, exist_ok=True)
        for f in files:
            fetch(f'{base}/{f}', f'{tmp}/{f}')
            sh(f'tar -xzf "{tmp}/{f}" -C "{tmp}" && rm "{tmp}/{f}"')
        sh(f'find "{tmp}" -type f -name "*.jpg" -exec mv -t "{jpg}" {{}} +')
        shutil.rmtree(tmp)
    for ds, base in [('roxford5k', 'oxford5k'), ('rparis6k', 'paris6k')]:
        d = f'{root}/test/{ds}'; os.makedirs(d, exist_ok=True)
        if not os.path.exists(f'{d}/jpg'): os.symlink(f'{root}/test/{base}/jpg', f'{d}/jpg')
        fetch(f'http://cmp.felk.cvut.cz/cnnimageretrieval/data/test/{ds}/gnd_{ds}.pkl', f'{d}/gnd_{ds}.pkl')
    for ds in ['oxford5k', 'paris6k']: print(ds, len(os.listdir(f'{root}/test/{ds}/jpg')), 'images')

def download_train(root):
    # same as cirtorch/utils/download.py (download_train), but extracts as a stream to avoid using double the disk space
    d = f'{root}/train/retrieval-SfM-120k'; ims = f'{d}/ims'; os.makedirs(ims, exist_ok=True)
    url = 'http://cmp.felk.cvut.cz/cnnimageretrieval/data/train/ims/ims.tar.gz'
    if not os.path.exists(f'{ims}/.done'):
        gb = remote_size_gb(url); free = shutil.disk_usage(root).free / 2**30
        print(f'ims.tar.gz ≈ {gb if gb is None else round(gb, 1)} GB, free {free:.1f} GB')
        if gb is not None and gb * 1.05 > free:
            raise RuntimeError('Not enough disk for rSfM-120k; change CFG["data_root"] or disable LEARN_WHITEN/RUN_STAGE2')
        sh(f'curl -L --retry 5 -sS "{url}" | tar -xzf - -C "{ims}" && touch "{ims}/.done"')
    for f in ['retrieval-SfM-120k.pkl', 'retrieval-SfM-120k-whiten.pkl']:
        fetch(f'http://cmp.felk.cvut.cz/cnnimageretrieval/data/train/dbs/{f}', f'{d}/{f}')
    print(disk(root))

t0 = time.time(); download_test(CFG['data_root']); print(f'test data ok ({time.time()-t0:.0f}s)')

## 2. Data, model, GeM

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # DINOv2 uses ImageNet normalization
PATCH = 14

def cid2filename(cid, prefix):
    return os.path.join(prefix, cid[-2:], cid[-4:-2], cid[-6:-4], cid)

def load_img(path, imsize, bbx=None):
    img = Image.open(path).convert('RGB')
    if bbx is not None: img = img.crop(tuple(bbx))           # query test: crop theo bbox
    img.thumbnail((imsize, imsize), Image.LANCZOS)            # as in cirtorch: downscale only, keep aspect ratio (LANCZOS instead of ANTIALIAS)
    w, h = img.size; w14, h14 = max(PATCH, w // PATCH * PATCH), max(PATCH, h // PATCH * PATCH)
    if (w14, h14) != (w, h):
        if w < PATCH or h < PATCH: img = img.resize((w14, h14), Image.BICUBIC)
        else:
            l, t = (w - w14) // 2, (h - h14) // 2; img = img.crop((l, t, l + w14, t + h14))  # center-crop < 14px
    x = torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.).permute(2, 0, 1)
    return (x - torch.tensor(MEAN)[:, None, None]) / torch.tensor(STD)[:, None, None]

class ImageList(Dataset):
    def __init__(self, paths, imsize, bbxs=None): self.paths, self.imsize, self.bbxs = paths, imsize, bbxs
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return load_img(self.paths[i], self.imsize, None if self.bbxs is None else self.bbxs[i])

class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__(); self.p = nn.Parameter(torch.ones(1) * p); self.eps = eps
    def forward(self, x):  # B x D x h x w  ->  B x D
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), x.shape[-2:]).pow(1. / self.p).flatten(1)

def flat_blocks(bb):
    out = []
    for b in bb.blocks:  # also supports BlockChunk
        out.extend([m for m in b if not isinstance(m, nn.Identity)] if isinstance(b, nn.ModuleList) else [b])
    return out

class B0Net(nn.Module):
    # DINOv2 patch tokens -> ReLU -> GeM -> L2   (Lw applied afterwards, outside the network, as in cirtorch)
    def __init__(self, backbone, p=3.0, finetune_blocks=0):
        super().__init__()
        self.backbone, self.pool = backbone, GeM(p)
        self.blocks = flat_blocks(backbone)
        self.n_frozen = len(self.blocks) - finetune_blocks
        for prm in backbone.parameters(): prm.requires_grad = False
        for b in self.blocks[self.n_frozen:]:
            for prm in b.parameters(): prm.requires_grad = True
        if finetune_blocks > 0:
            for prm in backbone.norm.parameters(): prm.requires_grad = True
        self.dim = backbone.embed_dim

    def forward(self, x, heads=False):
        # heads=True: also return 2 diagnostic descriptors (CLS token, average pooling without ReLU)
        bb = self.backbone; B, _, H, W = x.shape; h, w = H // PATCH, W // PATCH
        with torch.autocast('cuda', dtype=torch.float16, enabled=AMP):
            with torch.no_grad():                                  # frozen part: no graph stored
                t = bb.prepare_tokens_with_masks(x)
                for blk in self.blocks[:self.n_frozen]: t = blk(t)
            for blk in self.blocks[self.n_frozen:]: t = blk(t)
            t = bb.norm(t)
        nreg = getattr(bb, 'num_register_tokens', 0)
        pt = t[:, 1 + nreg:].float()                               # B x N x D
        fmap = pt.transpose(1, 2).reshape(B, -1, h, w)
        gem = F.normalize(self.pool(F.relu(fmap)), dim=-1)        # ReLU: ViT tokens can be negative
        if not heads: return gem
        return {'gem': gem, 'cls': F.normalize(t[:, 0].float(), dim=-1), 'avg': F.normalize(pt.mean(1), dim=-1)}

HEADS = ['gem', 'cls', 'avg']   # gem = main B0; cls, avg only for comparison

def load_backbone():
    bb = torch.hub.load('facebookresearch/dinov2', CFG['arch'])
    return bb.to(DEV).eval()

def trainable_params(net): return [p for p in net.parameters() if p.requires_grad]

In [ ]:
def resize14(img, s):
    H, W = img.shape[-2:]
    nh, nw = max(PATCH, round(H * s / PATCH) * PATCH), max(PATCH, round(W * s / PATCH) * PATCH)
    return F.interpolate(img, size=(nh, nw), mode='bilinear', align_corners=False, antialias=True)

@torch.no_grad()
def extract(net, paths, imsize, scales=(1,), bbxs=None, desc='extract', heads=False):
    # Returns dict scale -> tensor D x N (L2-normalized). Multi-scale pooling is done later, so single- and multi-scale need only one extraction.
    # heads=True: dict head -> (dict scale -> tensor)
    net.eval()
    dl = DataLoader(ImageList(paths, imsize, bbxs), batch_size=1, shuffle=False, num_workers=CFG['num_workers'])
    out = {h: {s: torch.zeros(net.dim, len(paths)) for s in scales} for h in HEADS}
    for i, img in enumerate(tqdm(dl, desc=desc, mininterval=10)):
        img = img.to(DEV, non_blocking=True)
        for s in scales:
            o = net(img if s == 1 else resize14(img, s), heads=True)
            for h in HEADS: out[h][s][:, i] = o[h].squeeze(0).cpu()
    del dl; gc.collect()   # clean up old workers before opening a new DataLoader (avoids shutdown warnings)
    return out if heads else out['gem']

def combine_ms(per_scale, p):
    # as in cirtorch extract_ms: l2n( mean_s(v_s^p)^(1/p) ); p=1 (cls/avg can be negative) -> plain mean
    v = torch.stack([per_scale[s] for s in per_scale])
    v = v.mean(0) if p == 1 else v.clamp(min=0).pow(p).mean(0).pow(1. / p)
    return F.normalize(v, dim=0)

def variants(per_scale, p):
    return {'ss': per_scale[1], 'ms': combine_ms(per_scale, p)}

def msp(head, p): return p if head == 'gem' else 1

## 3. Evaluation (revisitop: Easy / Medium / Hard)

In [ ]:
def compute_ap(ranks, nres):
    ap, step = 0., 1. / nres
    for j, r in enumerate(ranks):
        p0 = 1. if r == 0 else j / r
        p1 = (j + 1) / (r + 1)
        ap += (p0 + p1) * step / 2.
    return ap

def compute_map(ranks, gnd, kappas=(1, 5, 10)):
    # ported from cirtorch/utils/evaluate.py (np.in1d -> np.isin)
    nq = len(gnd); aps = np.zeros(nq); prs = np.zeros((nq, len(kappas))); nempty = 0
    for i in range(nq):
        qgnd = np.array(gnd[i]['ok'])
        if qgnd.shape[0] == 0:
            aps[i] = np.nan; prs[i] = np.nan; nempty += 1; continue
        qgndj = np.array(gnd[i].get('junk', []))
        pos = np.arange(ranks.shape[0])[np.isin(ranks[:, i], qgnd)]
        junk = np.arange(ranks.shape[0])[np.isin(ranks[:, i], qgndj)]
        k = ij = 0
        if len(junk):
            for ip in range(len(pos)):
                while ij < len(junk) and pos[ip] > junk[ij]: k += 1; ij += 1
                pos[ip] -= k
        aps[i] = compute_ap(pos, len(qgnd))
        pos = pos + 1
        for j, kk in enumerate(kappas):
            kq = min(max(pos), kk); prs[i, j] = (pos <= kq).sum() / kq
    n = nq - nempty
    return np.nansum(aps) / n, np.nansum(prs, axis=0) / n

def eval_ranks(ds, ranks, gnd):
    out = {}
    for name, ok, jk in [('E', ['easy'], ['junk', 'hard']), ('M', ['easy', 'hard'], ['junk']), ('H', ['hard'], ['junk', 'easy'])]:
        g = [{'ok': np.concatenate([q[k] for k in ok]), 'junk': np.concatenate([q[k] for k in jk])} for q in gnd]
        m, pr = compute_map(ranks, g)
        out[name] = dict(mAP=round(100 * m, 2), **{f'mP@{k}': round(100 * v, 2) for k, v in zip((1, 5, 10), pr)})
    return out

def load_gnd(ds):
    with open(f"{CFG['data_root']}/test/{ds}/gnd_{ds}.pkl", 'rb') as f: return pickle.load(f)

def extract_testset(net, ds, scales, tag):
    cfg = load_gnd(ds); d = f"{CFG['data_root']}/test/{ds}/jpg"
    db = extract(net, [f'{d}/{n}.jpg' for n in cfg['imlist']], CFG['test_imsize'], scales, desc=f'{tag} {ds} db', heads=True)
    q = extract(net, [f'{d}/{n}.jpg' for n in cfg['qimlist']], CFG['test_imsize'], scales,
                bbxs=[g['bbx'] for g in cfg['gnd']], desc=f'{tag} {ds} query', heads=True)
    torch.save({'db': db, 'q': q}, f"{CFG['out_dir']}/vecs_{tag}_{ds}.pt")
    return db, q

RESULTS = []
def score_all(tag, vecs, p, whiten=None):
    # vecs: {ds: (db_heads, q_heads)}; whiten: {head: {'ss': (m,P), 'ms': (m,P)}} or None
    for ds, (dbh, qh) in vecs.items():
      gnd = load_gnd(ds)['gnd']
      for head in HEADS:
        dbv, qv = variants(dbh[head], msp(head, p)), variants(qh[head], msp(head, p))
        for sc in ['ss', 'ms']:
            for lw in ([False, True] if whiten else [False]):
                X, Q = dbv[sc].numpy(), qv[sc].numpy()
                if lw: X, Q = whitenapply(X, *whiten[head][sc]), whitenapply(Q, *whiten[head][sc])
                ranks = np.argsort(-(X.T @ Q), axis=0)
                r = eval_ranks(ds, ranks, gnd)
                key = (tag, ds, head, sc, lw)
                row = dict(stage=tag, dataset=ds, head=head, scale=sc, Lw=lw, p=round(float(p), 3),
                           **{f'{k}_{m}': v for k, d in r.items() for m, v in d.items()})
                RESULTS[:] = [x for x in RESULTS if (x['stage'], x['dataset'], x['head'], x['scale'], x['Lw']) != key] + [row]
                print(f"{tag:12s} {ds:9s} {head:3s} {sc} Lw={int(lw)} | mAP E {r['E']['mAP']:5.2f}  M {r['M']['mAP']:5.2f}  H {r['H']['mAP']:5.2f}"
                      f" | mP@10 M {r['M']['mP@10']:5.2f}  H {r['H']['mP@10']:5.2f}")
    save_results()

def save_results():
    import pandas as pd
    json.dump(RESULTS, open(f"{CFG['out_dir']}/results.json", 'w'), indent=1)
    pd.DataFrame(RESULTS).to_csv(f"{CFG['out_dir']}/results.csv", index=False)

## 4. Learned whitening Lw (Radenović, ported from `cirtorch/utils/whiten.py`)

In [ ]:
def cholesky(S):
    alpha = 0
    while True:
        try: return np.linalg.cholesky(S + alpha * np.eye(*S.shape))
        except np.linalg.LinAlgError: alpha = 1e-10 if alpha == 0 else alpha * 10

def whitenlearn(X, qidxs, pidxs):
    m = X[:, qidxs].mean(axis=1, keepdims=True)
    df = X[:, qidxs] - X[:, pidxs]
    S = df @ df.T / df.shape[1]
    P = np.linalg.inv(cholesky(S))
    df = P @ (X - m)
    eigval, eigvec = np.linalg.eigh(df @ df.T)
    order = eigval.argsort()[::-1]
    return m, eigvec[:, order].T @ P

def whitenapply(X, m, P):
    X = P @ (X - m)
    return X / (np.linalg.norm(X, axis=0, keepdims=True) + 1e-6)

def learn_whitening(net, scales, tag):
    d = f"{CFG['data_root']}/train/retrieval-SfM-120k"
    db = pickle.load(open(f'{d}/retrieval-SfM-120k-whiten.pkl', 'rb'))
    qi, pi = np.array(db['qidxs']), np.array(db['pidxs'])
    sel = np.random.RandomState(0).permutation(len(qi))[:CFG['whiten_max_pairs']]
    qi, pi = qi[sel], pi[sel]
    used = np.unique(np.concatenate([qi, pi])); remap = {int(o): n for n, o in enumerate(used)}
    paths = [cid2filename(db['cids'][i], f'{d}/ims') for i in used]
    print(f'Lw: {len(qi)} pairs, {len(paths)} images')
    per_head = extract(net, paths, CFG['test_imsize'], scales, desc=f'{tag} whiten', heads=True)
    p = net.pool.p.item(); q2, p2 = [remap[int(i)] for i in qi], [remap[int(i)] for i in pi]
    W = {}
    for head in HEADS:
        v = variants(per_head[head], msp(head, p))
        W[head] = {sc: whitenlearn(v[sc].numpy().astype(np.float64), q2, p2) for sc in v}
    pickle.dump(W, open(f"{CFG['out_dir']}/whiten_{tag}.pkl", 'wb'))
    return W

## 5. Stage 1: B0-frozen (no training)

In [ ]:
if RUN_STAGE1:
    net = B0Net(load_backbone(), p=CFG['gem_p'], finetune_blocks=0).to(DEV)
    t0 = time.time()
    vecs1 = {ds: extract_testset(net, ds, CFG['multiscale'], 'B0-frozen') for ds in CFG['datasets']}
    print(f'extract: {(time.time()-t0)/60:.1f} min')
    score_all('B0-frozen', vecs1, net.pool.p.item())

In [ ]:
if LEARN_WHITEN or RUN_STAGE2:
    t0 = time.time(); download_train(CFG['data_root']); print(f'train data ok ({(time.time()-t0)/60:.1f} min)')

if RUN_STAGE1 and LEARN_WHITEN:
    W1 = learn_whitening(net, CFG['multiscale'], 'B0-frozen')
    score_all('B0-frozen', vecs1, net.pool.p.item(), W1)

## 6. Stage 2: B0-finetune (contrastive + hard-negative mining, as in the original GeM)

In [ ]:
class Tuples:
    # compact port of cirtorch/datasets/traindataset.py (TuplesDataset)
    def __init__(self, split='train'):
        d = f"{CFG['data_root']}/train/retrieval-SfM-120k"
        db = pickle.load(open(f'{d}/retrieval-SfM-120k.pkl', 'rb'))[split]
        self.images = [cid2filename(c, f'{d}/ims') for c in db['cids']]
        self.clusters, self.qpool, self.ppool = db['cluster'], db['qidxs'], db['pidxs']
        print(f'{split}: {len(self.images)} images, {len(self.qpool)} q/p pairs')

    def mine(self, net):
        sel = torch.randperm(len(self.qpool))[:CFG['query_size']]
        self.qidxs = [self.qpool[i] for i in sel]; self.pidxs = [self.ppool[i] for i in sel]
        pool = torch.randperm(len(self.images))[:CFG['pool_size']].tolist()
        qv = extract(net, [self.images[i] for i in self.qidxs], CFG['train_imsize'], desc='mine q')[1]
        pv = extract(net, [self.images[i] for i in pool], CFG['train_imsize'], desc='mine pool')[1]
        ranks = torch.argsort(pv.t() @ qv, dim=0, descending=True)
        self.nidxs, dsum = [], 0.
        for q in range(len(self.qidxs)):
            clusters, nid, r = {self.clusters[self.qidxs[q]]}, [], 0
            while len(nid) < CFG['neg_num'] and r < len(pool):
                cand = pool[ranks[r, q]]
                if self.clusters[cand] not in clusters:               # different cluster, one image per cluster
                    nid.append(cand); clusters.add(self.clusters[cand])
                    dsum += (qv[:, q] - pv[:, ranks[r, q]]).norm().item()
                r += 1
            self.nidxs.append(nid)
        return dsum / max(1, sum(len(n) for n in self.nidxs))

class TupleSet(Dataset):
    def __init__(self, T): self.T = T
    def __len__(self): return len(self.T.qidxs)
    def __getitem__(self, i):
        idx = [self.T.qidxs[i], self.T.pidxs[i]] + self.T.nidxs[i]
        return [load_img(self.T.images[j], CFG['train_imsize']) for j in idx]

def contrastive_loss(x, margin, eps=1e-6):
    # x: D x (2+nneg) = [q, p, n1..nk]; as in cirtorch/layers/functional.py
    d = (x[:, :1] - x[:, 1:] + eps).pow(2).sum(0).sqrt()
    lbl = torch.zeros_like(d); lbl[0] = 1
    return (0.5 * lbl * d.pow(2) + 0.5 * (1 - lbl) * (margin - d).clamp(min=0).pow(2)).sum()

In [ ]:
def quick_eval(net, tag):
    db, q = extract_testset(net, 'roxford5k', (1,), tag)
    X, Q = db['gem'][1].numpy(), q['gem'][1].numpy()
    r = eval_ranks('roxford5k', np.argsort(-(X.T @ Q), axis=0), load_gnd('roxford5k')['gnd'])
    return r['M']['mAP'], r['H']['mAP']

def train_stage2():
    net = B0Net(load_backbone(), p=CFG['gem_p'], finetune_blocks=CFG['finetune_blocks']).to(DEV)
    body = [p for n, p in net.named_parameters() if p.requires_grad and not n.startswith('pool.')]
    opt = torch.optim.Adam([{'params': body},
                            {'params': net.pool.parameters(), 'lr': CFG['lr'] * CFG['p_lr_mult'], 'weight_decay': 0}],
                           lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    sched = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=CFG['lr_gamma'])
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    print(f"trainable params: {sum(p.numel() for p in trainable_params(net))/1e6:.1f}M "
          f"(last {CFG['finetune_blocks']} / {len(net.blocks)} blocks + norm + p)")
    start, log = 0, []
    ck = CFG['resume_path'] or f"{CFG['out_dir']}/ckpt_last.pt"
    if os.path.exists(ck):
        s = torch.load(ck, map_location=DEV)
        net.load_state_dict(s['trainable'], strict=False); opt.load_state_dict(s['opt']); sched.load_state_dict(s['sched'])
        start, log = s['epoch'], s['log']; print(f'resumed from {ck}, epoch {start}')

    T = Tuples('train'); t_begin = time.time(); ep_time = 0.
    for ep in range(start, CFG['epochs']):
        if (time.time() - t_begin + ep_time) / 3600 > CFG['max_train_hours']:
            print('Early stop: about to exceed max_train_hours'); break
        t0 = time.time()
        avg_nd = T.mine(net)
        net.train()
        dl = DataLoader(TupleSet(T), batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'],
                        collate_fn=lambda b: b, drop_last=True)
        tot, n = 0., 0
        for batch in tqdm(dl, desc=f'epoch {ep+1}', mininterval=30):
            opt.zero_grad(set_to_none=True)
            for tup in batch:
                vecs = torch.cat([net(im.unsqueeze(0).to(DEV, non_blocking=True)) for im in tup]).t()  # D x 7
                loss = contrastive_loss(vecs, CFG['margin'])
                scaler.scale(loss).backward(); tot += loss.item(); n += 1
            scaler.step(opt); scaler.update()
        sched.step()
        ep_time = time.time() - t0
        rec = dict(epoch=ep + 1, loss=tot / max(1, n), avg_neg_dist=avg_nd, p=net.pool.p.item(), min=ep_time / 60)
        if CFG['eval_every_epoch']: rec['roxf_M'], rec['roxf_H'] = quick_eval(net, 'tmp')
        log.append(rec); print(rec)
        names = {n_ for n_, p in net.named_parameters() if p.requires_grad}
        sd = {k: v for k, v in net.state_dict().items() if k in names}
        torch.save(dict(trainable=sd, opt=opt.state_dict(), sched=sched.state_dict(), epoch=ep + 1, log=log, cfg=CFG),
                   f"{CFG['out_dir']}/ckpt_last.pt")
        json.dump(log, open(f"{CFG['out_dir']}/train_log.json", 'w'), indent=1)
    return net, log

if RUN_STAGE2:
    if 'net' in globals(): del net; torch.cuda.empty_cache()
    net2, log2 = train_stage2()

In [ ]:
if RUN_STAGE2:
    vecs2 = {ds: extract_testset(net2, ds, CFG['multiscale'], 'B0-finetune') for ds in CFG['datasets']}
    score_all('B0-finetune', vecs2, net2.pool.p.item())
    if LEARN_WHITEN:
        W2 = learn_whitening(net2, CFG['multiscale'], 'B0-finetune')
        score_all('B0-finetune', vecs2, net2.pool.p.item(), W2)

## 7. Results table

In [ ]:
import pandas as pd
df = pd.DataFrame(RESULTS)
if len(df):
    piv = df.pivot_table(index=['stage', 'head', 'scale', 'Lw'], columns='dataset', values=['E_mAP', 'M_mAP', 'H_mAP'])
    piv = piv.reorder_levels([1, 0], axis=1).sort_index(axis=1)
    display(piv.round(2))
    print('Reference: DINOv2 paper ViT-B/14 frozen  ROxf M/H 72.9/49.5 | RPar M/H 90.3/78.6')
    print('                GeM R101 (cirtorch, ms + Lw)   ROxf M/H 65.4/40.1 | RPar M/H 76.7/55.2')
print('Files:', sorted(os.listdir(CFG['out_dir'])))